### RAG Pipline - Data ingestion to Vector DB Pipeline


In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf files inside the directory


def process_all_pdfs(pdf_directory):

  all_documents = []
  pdf_dir = Path(pdf_directory)

  pdf_files = list(pdf_dir.glob("**/*.pdf"))

  for pdf_file in pdf_files:
    print(f"\nprocessing: {pdf_file.name}")
    try:
      loader = PyMuPDFLoader(str(pdf_file))
      documents = loader.load()

      for doc in documents:
        doc.metadata['source_file'] = pdf_file.name
        doc.metadata['file_type'] = 'pdf'

      all_documents.extend(documents)

    except Exception as e:
      print(f" x Error: {e}")

  return all_documents

all_pdf_documents = process_all_pdfs("../data")


processing: c_intro.pdf

processing: java_intro.pdf

processing: python_intro.pdf

processing: rust_intro.pdf


In [5]:
all_pdf_documents

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\c_intro.pdf', 'file_path': '..\\data\\pdf\\c_intro.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'c_intro.pdf', 'file_type': 'pdf'}, page_content="C Programming\nOverview\nC is a procedural programming language that provides direct control over memory and hardware\nresources. It has influenced many later languages and remains important for operating systems,\nembedded systems, and performance-sensitive software.\nVariables and Memory\nC uses statically declared types such as int, float, double, and char. Variables occupy memory, and\nprogrammers can use pointers to store and manipulate memory addresses.\nPointers\nPointers are one of C's defining features. A pointer can refer to another object's memory address,\nallowing functions and data struc

In [6]:
### Text Splitting get into chunks

def split_documents(documents, chunk_size = 500, chunk_overlap=200):
  text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
  )

  split_docs = text_splitter.split_documents(documents)
  print(f"split {len(documents)} documents into {len(split_docs)} chunks")

  # example of a chunk
  if split_docs:
    print(f"\nExample chunk:")
    print(f"Content: {split_docs[0].page_content[:200]}...")

  return split_docs


In [8]:
chunks = split_documents(all_pdf_documents)
chunks

split 4 documents into 12 chunks

Example chunk:
Content: C Programming
Overview
C is a procedural programming language that provides direct control over memory and hardware
resources. It has influenced many later languages and remains important for operatin...


[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\c_intro.pdf', 'file_path': '..\\data\\pdf\\c_intro.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'c_intro.pdf', 'file_type': 'pdf'}, page_content='C Programming\nOverview\nC is a procedural programming language that provides direct control over memory and hardware\nresources. It has influenced many later languages and remains important for operating systems,\nembedded systems, and performance-sensitive software.\nVariables and Memory\nC uses statically declared types such as int, float, double, and char. Variables occupy memory, and\nprogrammers can use pointers to store and manipulate memory addresses.\nPointers'),
 Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\c_intro.pdf', 'file_path': '..\\da

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:
  def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

    self.model_name = model_name
    self.model = None
    self._load_model()

  def _load_model(self): #loads the sentence transformer model

    try:
      print("loading embedding model")
      self.model = SentenceTransformer(self.model_name)
      print("Model loaded successfully")
    except Exception as e:
      print("Error loading model {e}")
      raise

  def generate_embeddings(self, texts: List[str]) -> np.ndarray: #generate embeddings for a list of texts
    """
      generate embeddings for a list of texts


      args:
        texts: list of text strings to embed
    """
    if not self.model:
      raise ValueError("Model not loaded")

    print(f"generating embeddings for {len(texts)}")
    embeddings = self.model.encode(texts, show_progress_bar=True)
    print(f"generated embeddings with shape: {embeddings.shape}")
    return embeddings


  def get_embedding_dimension(self) -> int:
    if not self.model:
      raise ValueError("Model is not loaded")
    return self.model.get_embedding_dimension()


embedding_manager = EmbeddingManager()
embedding_manager


loading embedding model


c:\Users\Hisen\Documents\Personal Projects\Full-Stack-Guide\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hisen\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6636.15i

Model loaded successfully


### VectorStore

In [ ]:
class VectorStore:


  def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):

    """
      initialize the vector storage

      Args:
      collection_name: Name of the ChromaDB collection
      persist_directory: Directory to persist the vector store


    """

    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.client = None
    self.collection = None
    self._initialize_store()

  def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
          # get or create collection
          os.makedirs(self.persist_directory, exist_ok=True)
          self.client = chromadb.PersistentClient(path=self.persist_directory)

          self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata={"description": "PDF document embeddings for RAG"}
          )

        except Exception as e:
          print(f"Error initializing vector store: {e}")
          raise

  def add_documents(self, documents: List[any], embeddings: np.ndarray):
    """
    Add documents and their embeddings to the vector store

    args:
      documents: list of langchain documents
      embeddings: corresponding embeddings for the documents
    """

    if len(documents) != len(embeddings):
      raise ValueError("Number of documents must match number of embeddings")


    # prep data for chromaDB
    ids = []
    metadatas = []
    documents_text = []
    embeddings_list = []


    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

      # generate unique ID
      doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
      ids.append(doc_id)

      #prepare metadata
      metadata = dict(doc.metadata)
      metadata['doc_index'] = i
      metadata['content_length'] = len(doc.page_content)
      metadatas.append(metadata)

      documents_text.append(doc.page_content)

      embeddings_list.append(embedding.tolist())

      try:
        self.collection.add(
          ids=ids,
          embeddings=embeddings_list,
          metadatas=metadatas,
          documents=documents_text
        )

      except Exception as e:
        print(f"error adding documents to vector store: {e}")
        raise


vectorstore = VectorStore()
vectorstore

In [ ]:
### convert the text to embeddings

texts = [doc.page_content for doc in chunks]

### generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)

## store in the vector database
vectorstore.add_documents(chunks, embeddings)


generating embeddings for 12


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.44it/s]


generated embeddings with shape: (12, 384)


### Retriever Piprline From VectorStore

In [ ]:
class RAGRetriever:
  """Handles query-based retrieval from the vector store"""

  def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager)